### Pydantic 格式使用

#### 1. 基本使用

举例1: 

In [5]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [6]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    """
    人物信息
    """
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

result = structured_model.invoke("张三是一名30岁的软件工程师")

rprint(result)
rprint(type(result))

Person(name='张三', age=30, occupation='软件工程师')

<class '__main__.Person'>

举例2:

In [7]:
class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="发行年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分, 满分十分")

structured_model = model.with_structured_output(MovieModel)

result = structured_model.invoke("给出电脑盗梦空间的信息")
print(result)

title='关于“电脑盗梦空间”的信息' year=2010 director='克里斯托弗·诺兰' rating=9.4


#### 2. 高级特性

##### 2.1 情况1: 可选字段

举例

In [ ]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    """
    人物信息
    """
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

rprint(result)
rprint(type(result))

作为对比

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class Person(BaseModel):
    """
    人物信息
    """
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

rprint(result)
rprint(type(result))

### 2.2 情况2: 默认值

不同的模型供应商, 对于此字段的支持是不同的, 比如: closeai 平台的 gpt-5.3-mini 模型就不支持此字段, 而 openrouter 平台的 gpt-5.4-mini 模型就支持此字段

In [ ]:
# closeai 的模型

import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model_with_closeai = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class Person(BaseModel):
    """
    人物信息
    """
    name: str = Field(description="姓名")
    age: int = Field(default=10, description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model_with_closeai.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

rprint(result)
rprint(type(result))

In [ ]:
# openrouter 的模型

import os

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model_with_openrouter = ChatOpenRouter(
    model='openai/glm-5.3',
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class Person(BaseModel):
    """
    人物信息
    """
    name: str = Field(description="姓名")
    age: int = Field(default=10, description="年龄")
    occupation: str = Field(description="职业")

# 创建结构化输出的大语言模型
structured_model = model_with_openrouter.with_structured_output(Person)

result = structured_model.invoke("张三是一名软件工程师")

rprint(result)
rprint(type(result))

#### 2.3 枚举类型

问题: 如何限制字段的可选值

回答: 使用枚举

举例1:

In [ ]:
from enum import Enum

class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class Task(BaseModel):
    title: str
    priority: Priority  # 只能是 LOW/MEDIUM/HIGH

#### 2.4 情况4: 列表提取

举例1:

In [10]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [11]:
from typing import List

class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")

class PersonList(BaseModel):
    """
    人物列表
    """
    people: List[Person] # 多个 Person 的对象

structured_model = model.with_structured_output(PersonList)

result = structured_model.invoke("张三 30岁, 李四 40岁")
print(result)

people=[Person(name='张三', age=30), Person(name='李四', age=40)]


举例2:

In [12]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")

structured_llm = model.with_structured_output(Review)

review = structured_llm.invoke(
    """iPhone 17 很棒! 摄像头强大, 手感好, 但是价格贵, 没有充电器. 4分"""
)
rprint(review)

Review(product='iPhone 17', rating=4, pros=['摄像头强大', '手感好'], cons=['价格贵', '没有充电器'])

#### 2.5 嵌套结构

举例1:

In [14]:
class Address(BaseModel):
    """地点描述"""
    city: str = Field(description="城市")
    district: str = Field(description="区域")

class Company(BaseModel):
    """公司信息"""
    name: str = Field(description="公司名称")
    address: Address = Field(description="公司所在地")

structured_llm = model.with_structured_output(Company)

result = structured_llm.invoke("a在杭州的滨江区")
rprint(result)

Company(name='a', address=Address(city='杭州', district='滨江区'))

#### 2.6 限制条件

In [15]:
class User(BaseModel):
    name: str = Field(description="姓名", min_length=2, max_length=50)
    age: int = Field(description="年龄", le=150)
    email: str = Field(description="邮箱")

user1 = User(name="tom", age=20, email="tom@125.com")
print(user1)

name='tom' age=20 email='tom@125.com'


In [17]:
from pydantic import ValidationError
try:
    user2 = User(name="tom", age=200, email="tom@126.com")
    print(user2)
except ValidationError as e:
    print(f"[FAIL]{e}")

[FAIL]1 validation error for User
age
  Input should be less than or equal to 150 [type=less_than_equal, input_value=200, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal
